# 07 — E2 shortcut audit, test-time ablation (Apratim Sharma)

**Question.** The data is already format-normalised (16 kHz mono, 160,000
samples), so sample rate and length cannot separate the classes. What can
still separate them is *resampling history*: the authentic clips were
resampled 48 kHz -> 16 kHz by our pipeline, Stable Audio Open generates at
44.1 kHz and MusicGen at 32 kHz before FakeMusicCaps resampled them, and
audioldm2 / musicldm / mustango generate natively at 16 kHz. Each route
leaves a different signature in the band just below Nyquist (8 kHz). If the
models lean on that band, removing it should hurt them far more than it
hurts a listener (locked decision 6).

**Protocol.** No retraining. The already-trained M1, M2, M3 (trained on all
five generators, the §6 table) are evaluated on the 1,604-clip test split
under progressively stronger perturbations applied to the raw waveform of
BOTH classes, then passed through the exact M1/M2 feature code:
- intact (recomputed from raw audio; must reproduce the cache),
- low-pass at 7, 6, 5, 4, 3, 2 kHz (zero-phase Butterworth),
- MP3 re-encode at 128, 64, 32 kbps (libmp3lame, decoded back to 16 kHz).

**Prediction, written before the run.** If the cue sits near Nyquist,
balanced accuracy falls by several points already at 7 kHz, where almost no
musical content is lost. Following Sroka et al. (2025, ref. 2), who found
that corrupting high frequencies pushes detectors towards "fake", the
failure should show up mainly as REAL clips being flagged synthetic
(specificity collapsing), not as synthetic recall falling. If instead
accuracy degrades gently and symmetrically, the near-Nyquist band is not the
main cue. Either outcome is reported.

In [ ]:
# Environment and inputs.
import os, glob, json, time, shutil, subprocess, tempfile
import numpy as np, pandas as pd, torch

print(torch.__version__, '| cuda:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
assert torch.cuda.is_available(), 'no GPU attached to this kernel'

def find_input(name, marker, required=True):
    """Inputs mount at /kaggle/input/datasets/<user>/<name>/; fall back to a
    search so a layout change gives a clear message."""
    d = f'/kaggle/input/datasets/akshatyuvan/{name}'
    if os.path.exists(f'{d}/{marker}'):
        return d
    hits = glob.glob(f'/kaggle/input/**/{name}/**/{marker}', recursive=True)
    if not hits:
        assert not required, f'{name}/{marker} not found under /kaggle/input'
        return None
    return os.path.dirname(hits[0])

MELS_DIR = find_input('fmc-m2-mels', 'mels.npy')
M1_DIR   = find_input('fmc-m1-features', 'X.npy')
M3_DIR   = find_input('fmc-m3-results', 'm3_best.pt')
RAW_DIR  = find_input('fakemusiccaps-raw', 'mustango')
REAL_DIR = find_input('musiccaps-real-16k', '-0Gj8-vB1q4.wav')
DONE_DIR = find_input('fmc-e2-results', 'README.md', required=False)
for d in [MELS_DIR, M1_DIR, M3_DIR, RAW_DIR, REAL_DIR, DONE_DIR]:
    print(d)

OUT, SEED = '/kaggle/working', 42

# libmp3lame is needed for the re-encode conditions. The Kaggle image's ffmpeg
# normally has it; if not, imageio-ffmpeg ships a static build that does.
def find_ffmpeg():
    for exe in [shutil.which('ffmpeg')]:
        if exe and 'libmp3lame' in subprocess.run([exe, '-hide_banner', '-encoders'],
                                                  capture_output=True, text=True).stdout:
            return exe
    subprocess.run(['pip', 'install', '-q', 'imageio-ffmpeg'], check=True)
    import imageio_ffmpeg
    return imageio_ffmpeg.get_ffmpeg_exe()
FFMPEG = find_ffmpeg()
print('ffmpeg:', FFMPEG)

In [ ]:
# The test split of the same 10,697-clip subset every model was evaluated on.
sub    = pd.read_csv(f'{MELS_DIR}/subset.csv', dtype={'ytid': str})
M      = np.load(f'{MELS_DIR}/mels.npy', mmap_mode='r')
labels = np.load(f'{MELS_DIR}/labels.npy')
X1     = np.load(f'{M1_DIR}/X.npy')
sub1   = pd.read_csv(f'{M1_DIR}/subset.csv', dtype={'ytid': str})
for c in ['ytid', 'generator', 'label', 'split']:
    assert (sub[c].values == sub1[c].values).all(), f'M1/M2 subsets disagree on {c}'
assert (sub.label.values == labels).all()

# Raw-audio paths, remapped exactly as the M2 notebook did (cell 1).
sub['path'] = (sub.path
    .str.replace('/kaggle/working/real', REAL_DIR, regex=False)
    .str.replace('/kaggle/input/datasets/akshatyuvan/fakemusiccaps-raw', RAW_DIR, regex=False))

idx = {s: np.where((sub.split == s).values)[0] for s in ['train', 'val', 'test']}
te = idx['test']
y_te, gen_te = labels[te], sub.generator.values[te]
assert len(te) == 1604 and all(map(os.path.exists, sub.path.values[te]))
print(pd.Series(gen_te).value_counts().to_dict())

In [ ]:
# Feature code: the bodies of M2's mel() and M1's feats(), verbatim, split so
# they take a waveform instead of a path. Perturbations act on the waveform
# in between reading and featurising.
import soundfile as sf, librosa
from scipy.signal import butter, sosfiltfilt, correlate
from joblib import Parallel, delayed

SR, N_MELS, N_FFT, HOP = 16000, 128, 1024, 160
N_SAMPLES = 160000
N_FRAMES  = N_SAMPLES // HOP + 1

def fix_len(y):
    return y[:N_SAMPLES] if len(y) >= N_SAMPLES else np.pad(y, (0, N_SAMPLES - len(y)))

def load(path):
    y, sr = sf.read(path, dtype='float32')
    assert sr == SR
    return fix_len(y)

def mel_y(y):                          # M2 notebook, cell 2, after reading
    S = librosa.feature.melspectrogram(y=y, sr=SR, n_fft=N_FFT,
                                       hop_length=HOP, n_mels=N_MELS)
    S = librosa.power_to_db(S, ref=np.max)
    return S[:, :N_FRAMES].astype(np.float16)

def feats_y(y, sr=SR):                 # M1 notebook, cell 6, after reading
    S = np.abs(librosa.stft(y, n_fft=2048, hop_length=512))
    blocks = [
        librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20),
        librosa.feature.chroma_stft(S=S, sr=sr),
        librosa.feature.spectral_contrast(S=S, sr=sr),
        librosa.feature.spectral_centroid(S=S, sr=sr),
        librosa.feature.spectral_rolloff(S=S, sr=sr),
        librosa.feature.spectral_bandwidth(S=S, sr=sr),
        librosa.feature.zero_crossing_rate(y),
    ]
    return np.concatenate([np.concatenate([b.mean(1), b.std(1)]) for b in blocks])

def lowpass(y, fc):
    # 8th-order Butterworth run forwards and backwards (sosfiltfilt): zero
    # phase, so the waveform is not shifted in time, and a steep enough
    # roll-off that the band above fc is genuinely removed.
    sos = butter(8, fc, btype='low', fs=SR, output='sos')
    return sosfiltfilt(sos, y).astype(np.float32)

def mp3(y, kbps):
    # Real files rather than pipes: with a seekable output libmp3lame writes
    # the LAME header, and ffmpeg's decoder uses it to drop the encoder delay,
    # so the decoded clip lines up with the original sample for sample.
    with tempfile.TemporaryDirectory() as d:
        sf.write(f'{d}/in.wav', y, SR, subtype='FLOAT')
        subprocess.run([FFMPEG, '-y', '-v', 'error', '-i', f'{d}/in.wav',
                        '-c:a', 'libmp3lame', '-b:a', f'{kbps}k', f'{d}/x.mp3'], check=True)
        subprocess.run([FFMPEG, '-y', '-v', 'error', '-i', f'{d}/x.mp3',
                        '-ac', '1', '-ar', str(SR), '-c:a', 'pcm_f32le', f'{d}/out.wav'], check=True)
        z, sr = sf.read(f'{d}/out.wav', dtype='float32')
    assert sr == SR
    return fix_len(z)

CONDITIONS = [('intact', None, None)] + \
             [(f'lowpass_{fc // 1000}k', 'lowpass', fc) for fc in [7000, 6000, 5000, 4000, 3000, 2000]] + \
             [(f'mp3_{k}k', 'mp3', k) for k in [128, 64, 32]]

def perturb(y, kind, arg):
    if kind is None:      return y
    if kind == 'lowpass': return lowpass(y, arg)
    if kind == 'mp3':     return mp3(y, arg)
    raise ValueError(kind)

def process(path, kind, arg):
    y = perturb(load(path), kind, arg)
    return mel_y(y), feats_y(y)

# MP3 alignment check: a constant offset between original and decoded audio
# would be a second, unintended perturbation. Measure the lag on 8 clips.
lags = []
for p in sub.path.values[te][:8]:
    y = load(p); z = mp3(y, 64)
    c = correlate(z[:48000], y[:48000], mode='full', method='fft')
    mid = len(c) // 2
    lags.append(int(np.argmax(c[mid - 2048: mid + 2049]) - 2048))
print('mp3 decode lag (samples) on 8 clips:', lags)
assert int(np.median(lags)) == 0, 'decoded MP3 is offset from the original; fix before trusting mp3 conditions'

In [ ]:
# The three trained models: M2 and M3 from their saved checkpoints; M1 had no
# checkpoint saved, so it is retrained here with the 03b rerun code (11 s),
# pinned to one CPU thread so this copy is bit-reproducible, and saved.
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (balanced_accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve)

class M2CNN(nn.Module):
    def __init__(self):
        super().__init__()
        def blk(i, o):
            return nn.Sequential(
                nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(),
                nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(),
                nn.MaxPool2d(2))
        self.features = nn.Sequential(blk(1,32), blk(32,64), blk(64,128), blk(128,256))
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                  nn.Dropout(0.3), nn.Linear(256,1))
    def forward(self, x): return self.head(self.features(x))

class M3CRNN(nn.Module):
    def __init__(self, n_mels=128, hidden=128):
        super().__init__()
        def blk(i, o):
            return nn.Sequential(
                nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(),
                nn.MaxPool2d(kernel_size=(2, 1)))
        self.features = nn.Sequential(blk(1, 32), blk(32, 64), blk(64, 128), blk(128, 128))
        self.rnn = nn.LSTM(input_size=128 * (n_mels // 16), hidden_size=hidden,
                           batch_first=True, bidirectional=True)
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(2 * hidden, 1))
    def forward(self, x):
        z = self.features(x)
        b, c, f, t = z.shape
        seq, _ = self.rnn(z.permute(0, 3, 1, 2).reshape(b, t, c * f))
        return self.head(seq.mean(dim=1))

def metrics(true, score):
    """Line-for-line the M1/M2/M3 metric code."""
    pred = (score > 0.5).astype(int)
    fpr, tpr, _ = roc_curve(true, score)
    eer = float(fpr[np.nanargmin(np.abs((1 - tpr) - fpr))])
    return {"balanced_accuracy": float(balanced_accuracy_score(true, pred)),
            "precision": float(precision_score(true, pred, zero_division=0)),
            "recall":    float(recall_score(true, pred)),
            "f1":        float(f1_score(true, pred)),
            "auc":       float(roc_auc_score(true, score)),
            "eer":       eer}

# ---- M1: the 03b rerun training cell, one thread.
torch.manual_seed(SEED); np.random.seed(SEED)
n_threads = torch.get_num_threads(); torch.set_num_threads(1)
sc = StandardScaler().fit(X1[idx['train']])             # train split only (decision 4)
T = lambda a: torch.tensor(sc.transform(a), dtype=torch.float32)
Xtr, Xva = T(X1[idx['train']]), T(X1[idx['val']])
ytr = torch.tensor(labels[idx['train']], dtype=torch.float32)
yva = torch.tensor(labels[idx['val']], dtype=torch.float32)
m1 = nn.Sequential(
    nn.Linear(86, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(128, 1))
opt = torch.optim.Adam(m1.parameters(), lr=1e-3, weight_decay=1e-4)
lossf = nn.BCEWithLogitsLoss()
best, best_ep, best_state, bad = 0.0, -1, None, 0
for ep in range(200):
    m1.train(); perm = torch.randperm(len(Xtr))
    for i in range(0, len(perm), 128):
        b = perm[i:i+128]; opt.zero_grad()
        lossf(m1(Xtr[b]).squeeze(1), ytr[b]).backward(); opt.step()
    m1.eval()
    with torch.no_grad():
        v = ((m1(Xva).squeeze(1) > 0).float() == yva).float().mean().item()
    if v > best:
        best, best_ep, bad = v, ep, 0
        best_state = {k: t.clone() for k, t in m1.state_dict().items()}
    else:
        bad += 1
        if bad >= 15: break
m1.load_state_dict(best_state); m1.eval()
torch.save(best_state, f'{OUT}/e2_m1_retrained.pt')
torch.set_num_threads(n_threads)
print(f'M1 retrained: best val {best:.4f} at epoch {best_ep}')

# ---- M2 / M3 from checkpoints; mel normalisation from the train split, as
# both notebooks computed it.
tr_mean = float(M[idx['train']].astype(np.float32).mean())
tr_std  = float(M[idx['train']].astype(np.float32).std())
m3_cfg  = json.load(open(f'{M3_DIR}/m3_results.json'))['config']
assert abs(tr_mean - m3_cfg['norm_mean']) < 1e-3 and abs(tr_std - m3_cfg['norm_std']) < 1e-3
m2 = M2CNN().cuda(); m2.load_state_dict(torch.load(f'{MELS_DIR}/m2_best.pt', map_location='cuda')); m2.eval()
m3 = M3CRNN().cuda(); m3.load_state_dict(torch.load(f'{M3_DIR}/m3_best.pt', map_location='cuda')); m3.eval()

def score_m1(F):
    with torch.no_grad():
        return torch.sigmoid(m1(T(F)).squeeze(1)).numpy()

def score_deep(model, mels):
    out = []
    with torch.no_grad():
        for i in range(0, len(mels), 32):
            x = (np.asarray(mels[i:i+32], dtype=np.float32) - tr_mean) / tr_std
            x = torch.from_numpy(x).unsqueeze(1).cuda()
            with torch.amp.autocast('cuda', dtype=torch.float16):
                out.append(torch.sigmoid(model(x).squeeze(1).float()).cpu().numpy())
    return np.concatenate(out)

def score_all(mels, F):
    return {'M1': score_m1(F), 'M2': score_deep(m2, mels), 'M3': score_deep(m3, mels)}

# Reference: the three models on the CACHED test features must reproduce the
# §6 table. This is what ties every E2 number to the published baselines.
ref_scores = score_all(M[te], X1[te])
ref = {m: metrics(y_te, s)["balanced_accuracy"] for m, s in ref_scores.items()}
published = {'M1': json.load(open(f'{M1_DIR}/m1_results.json'))['balanced_accuracy'],
             'M2': json.load(open(f'{MELS_DIR}/m2_results.json'))['balanced_accuracy'],
             'M3': json.load(open(f'{M3_DIR}/m3_results.json'))['balanced_accuracy']}
print('cached-feature balanced accuracy:', {m: round(v, 4) for m, v in ref.items()},
      '| published:', published)
# M2 and M3 are fixed checkpoints and must match. M1 is a retrain and may
# differ in the last digit (CPU thread count, see CLAUDE.md §6); its E2
# deltas are taken against its own intact score, so that is recorded, not fatal.
assert abs(ref['M2'] - published['M2']) < 2e-3 and abs(ref['M3'] - published['M3']) < 2e-3

In [ ]:
# The sweep. Results are written after every condition, and conditions
# already present in the attached fmc-e2-results dataset are reused.
RES = f'{OUT}/e2_results.json'
results = {}
if DONE_DIR and os.path.exists(f'{DONE_DIR}/e2_results.json'):
    results = json.load(open(f'{DONE_DIR}/e2_results.json'))['conditions']
    print('reusing conditions:', list(results))

header = {"experiment": "E2_shortcut_audit_test_time", "seed": SEED, "split": "test",
          "n_test": int(len(te)), "n_real": int((y_te == 0).sum()), "n_synth": int((y_te == 1).sum()),
          "models": {"M1": "retrained here with the 03b rerun code, 1 CPU thread (e2_m1_retrained.pt)",
                     "M2": "fmc-m2-mels/m2_best.pt", "M3": "fmc-m3-results/m3_best.pt"},
          "m1_retrain_best_val_acc": round(best, 4), "m1_retrain_best_epoch": best_ep,
          "cached_feature_bal_acc": {m: round(v, 4) for m, v in ref.items()},
          "published_bal_acc": published, "mp3_decode_lag_samples": lags,
          "ffmpeg": FFMPEG, "librosa_version": librosa.__version__,
          "lowpass_filter": "Butterworth order 8, sosfiltfilt (zero phase)",
          "mp3_codec": "libmp3lame CBR, decoded by ffmpeg to 16 kHz float"}

def summarise(true, score, gens):
    m = {k: round(v, 4) for k, v in metrics(true, score).items()}
    pred = (score > 0.5).astype(int)
    m["specificity_real"] = round(float((pred[true == 0] == 0).mean()), 4)
    m["recall_per_generator"] = {g: round(float(pred[gens == g].mean()), 4)
                                 for g in sorted(set(gens[true == 1]))}
    m["mean_score_real"]  = round(float(score[true == 0].mean()), 4)
    m["mean_score_synth"] = round(float(score[true == 1].mean()), 4)
    return m

for name, kind, arg in CONDITIONS:
    if name in results:
        print(f'{name}: reused'); continue
    t0 = time.time()
    out = Parallel(n_jobs=4)(delayed(process)(p, kind, arg) for p in sub.path.values[te])
    mels = np.stack([o[0] for o in out]); F = np.array([o[1] for o in out])
    assert not np.isnan(mels.astype(np.float32)).any() and not np.isnan(F).any()
    entry = {"kind": kind, "arg": arg, "feature_time_s": round(time.time() - t0, 1)}
    if kind is None:
        # Recomputing from raw audio must reproduce the cache, or every other
        # condition would differ from the baseline in two ways at once.
        mel_diff = float(np.abs(mels.astype(np.float32) - M[te].astype(np.float32)).max())
        D = np.abs(F - X1[te]) / (X1.std(0) + 1e-12)
        feat_diff = float(D.max())
        r, c = np.unravel_index(D.argmax(), D.shape)
        # v1 of this kernel stopped here with a 1e-3 SD threshold: over all
        # 1,604 clips the largest descriptor difference was 7.7e-3 SD (the
        # mels matched exactly). A difference that small cannot move the
        # model, which the decisive check below confirms directly: M1's
        # predictions on recomputed vs cached descriptors.
        s_new, s_old = score_m1(F), score_m1(X1[te])
        flips = int(((s_new > 0.5) != (s_old > 0.5)).sum())
        print(f'intact vs cache: mel max |diff| {mel_diff:.4f} dB | descriptor max {feat_diff:.2e} SDs '
              f'(feature {c}, clip {sub.ytid.values[te][r]} / {gen_te[r]}) | '
              f'M1 max score diff {np.abs(s_new - s_old).max():.2e}, prediction flips {flips}')
        assert mel_diff < 0.1 and feat_diff < 0.05 and flips <= 1, 'raw-audio pipeline does not reproduce the cache'
        entry["cache_check"] = {"mel_max_abs_diff_db": round(mel_diff, 5),
                                "descriptor_max_diff_in_sd": float(f'{feat_diff:.3e}'),
                                "descriptor_max_diff_feature_index": int(c),
                                "descriptor_max_diff_clip": f'{sub.ytid.values[te][r]} / {gen_te[r]}',
                                "m1_max_score_diff": float(f'{np.abs(s_new - s_old).max():.3e}'),
                                "m1_prediction_flips": flips}
    for m, s in score_all(mels, F).items():
        entry[m] = summarise(y_te, s, gen_te)
    results[name] = entry
    json.dump({**header, "conditions": results}, open(RES, 'w'), indent=2)
    print(f'{name:12s} ' + '  '.join(
        f'{m} bal {entry[m]["balanced_accuracy"]:.4f} spec {entry[m]["specificity_real"]:.3f} '
        f'rec {entry[m]["recall"]:.3f} auc {entry[m]["auc"]:.3f}' for m in ['M1', 'M2', 'M3'])
        + f'  ({time.time() - t0:.0f}s)', flush=True)

In [ ]:
# Summary table and figures, from e2_results.json only.
import matplotlib.pyplot as plt

R = json.load(open(RES))['conditions']
rows = []
for name, _, _ in CONDITIONS:
    for m in ['M1', 'M2', 'M3']:
        e = R[name][m]
        rows.append({"condition": name, "model": m, "bal_acc": e["balanced_accuracy"],
                     "delta_vs_intact": round(e["balanced_accuracy"] - R['intact'][m]["balanced_accuracy"], 4),
                     "specificity_real": e["specificity_real"], "recall_synth": e["recall"],
                     "auc": e["auc"], "eer": e["eer"], **{f'rec_{g}': v for g, v in e["recall_per_generator"].items()}})
summ = pd.DataFrame(rows)
summ.to_csv(f'{OUT}/e2_summary.csv', index=False)
pd.set_option('display.width', 250)
print(summ.to_string(index=False))

lp = ['intact'] + [c for c, k, _ in CONDITIONS if k == 'lowpass']
mp = ['intact'] + [c for c, k, _ in CONDITIONS if k == 'mp3']
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for ax, conds, title in [(axes[0], lp, 'low-pass cutoff'), (axes[1], mp, 'MP3 bitrate')]:
    for m in ['M1', 'M2', 'M3']:
        d = summ[summ.model == m].set_index('condition').loc[conds]
        l, = ax.plot(range(len(conds)), d.bal_acc, 'o-', label=f'{m} balanced acc')
        ax.plot(range(len(conds)), d.specificity_real, 's:', color=l.get_color(), alpha=.7,
                label=f'{m} real specificity')
    ax.set_xticks(range(len(conds)))
    ax.set_xticklabels([c.replace('lowpass_', '').replace('mp3_', '') for c in conds])
    ax.set_title(title); ax.axhline(0.5, color='grey', lw=.8, ls='--'); ax.grid(alpha=.3)
axes[0].set_ylabel('test split'); axes[1].legend(fontsize=7, loc='lower left')
plt.tight_layout(); plt.savefig(f'{OUT}/e2_degradation.png', dpi=150); plt.show()

gens = sorted(R['intact']['M2']['recall_per_generator'])
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6), sharey=True)
for ax, m in zip(axes, ['M1', 'M2', 'M3']):
    for g in gens:
        ax.plot(range(len(lp)), [R[c][m]['recall_per_generator'][g] for c in lp], 'o-', label=g)
    ax.plot(range(len(lp)), [R[c][m]['specificity_real'] for c in lp], 'k--', label='real (specificity)')
    ax.set_xticks(range(len(lp))); ax.set_xticklabels([c.replace('lowpass_', '') for c in lp])
    ax.set_title(f'{m}: per-class accuracy vs low-pass'); ax.grid(alpha=.3)
axes[0].set_ylabel('fraction correct'); axes[2].legend(fontsize=7)
plt.tight_layout(); plt.savefig(f'{OUT}/e2_per_generator_lowpass.png', dpi=150); plt.show()
print(sorted(os.listdir(OUT)))